In [1]:
import sys
sys.path.append('C:/Users/sliso/LastTradingProject_7052026/indicators')

In [ ]:
import numpy as np
import pandas as pd
import json
import random
from collections import defaultdict, Counter
from itertools import product
from tqdm import tqdm
import numba

# ----------------------------------------------------------------------
# 1. Signal decorator (must match your actual one)
# ----------------------------------------------------------------------
def signal(direction, signal_type="continuous", weight=1.0):
    def decorator(func):
        func._signal_meta = {
            'direction': direction,
            'type': signal_type,
            'weight': weight,
            'name': func.__name__
        }
        return func
    return decorator

# ----------------------------------------------------------------------
# 2. Example indicator classes – REPLACE WITH YOUR REAL ONES
#    Each must have .category and decorated methods.
# ----------------------------------------------------------------------
class RSI:
    category = "momentum"
    def __init__(self, data, period=14):
        self.data = data
        # normally you would compute RSI array here – we skip for demo
        # In real code, store the indicator array as e.g., self.rsi
    @signal(direction="long", signal_type="continuous", weight=1.0)
    def below_down_level_long(self):
        # replace with actual logic using self.data, return numpy array of -1/0/1
        return np.random.choice([-1,0,1], size=len(self.data))
    @signal(direction="short", signal_type="continuous", weight=1.0)
    def above_up_level_short(self):
        return np.random.choice([-1,0,1], size=len(self.data))
    @signal(direction="long", signal_type="discrete", weight=2.0)
    def cross_above_down_level_long(self):
        return np.random.choice([-1,0,1], size=len(self.data))
    @signal(direction="short", signal_type="discrete", weight=2.0)
    def cross_below_up_level_short(self):
        return np.random.choice([-1,0,1], size=len(self.data))
    @signal(direction="long", signal_type="discrete", weight=2.5)
    def rsi_divergence_long(self):
        return np.random.choice([-1,0,1], size=len(self.data))

class Stochastic:
    category = "momentum"
    def __init__(self, data):
        self.data = data
    @signal(direction="long", signal_type="continuous", weight=1.0)
    def k_below_down_level_long(self):
        return np.random.choice([-1,0,1], size=len(self.data))
    @signal(direction="short", signal_type="continuous", weight=1.0)
    def k_above_up_level_short(self):
        return np.random.choice([-1,0,1], size=len(self.data))
    @signal(direction="long", signal_type="discrete", weight=2.0)
    def kd_cross_below_down_level_long(self):
        return np.random.choice([-1,0,1], size=len(self.data))
    @signal(direction="short", signal_type="discrete", weight=2.0)
    def kd_cross_above_up_level_short(self):
        return np.random.choice([-1,0,1], size=len(self.data))

class ADX:
    category = "trend_strength"
    def __init__(self, data):
        self.data = data
    @signal(direction="both", signal_type="continuous", weight=1.0)
    def above_level_trend(self):
        return np.random.choice([-1,0,1], size=len(self.data))

class GarmanKlass:
    category = "volatility"
    def __init__(self, data):
        self.data = data
    @signal(direction="both", signal_type="continuous", weight=1.0)
    def high_volatility_regime(self):
        return np.random.choice([-1,0,1], size=len(self.data))

# ----------------------------------------------------------------------
# 3. Allowed patterns (full lists from earlier)
# ----------------------------------------------------------------------
PATTERNS_2_ENTRY_LONG = [
    (('momentum', 'long', 'continuous'), ('momentum', 'long', 'discrete')),
    (('momentum', 'long', 'continuous'), ('divergence', 'long', 'discrete')),
    (('volatility', 'both', 'continuous'), ('momentum', 'long', 'discrete')),
    (('volatility', 'both', 'continuous'), ('divergence', 'long', 'discrete')),
    (('trend_strength', 'both', 'continuous'), ('momentum', 'long', 'discrete')),
    (('trend_strength', 'both', 'continuous'), ('divergence', 'long', 'discrete')),
]
PATTERNS_3_ENTRY_LONG = [
    (('momentum', 'long', 'continuous'), ('momentum', 'long', 'continuous'), ('momentum', 'long', 'discrete')),
    (('momentum', 'long', 'continuous'), ('momentum', 'long', 'continuous'), ('divergence', 'long', 'discrete')),
    (('volatility', 'both', 'continuous'), ('trend_strength', 'both', 'continuous'), ('momentum', 'long', 'discrete')),
    (('volatility', 'both', 'continuous'), ('momentum', 'long', 'continuous'), ('momentum', 'long', 'discrete')),
    (('trend_strength', 'both', 'continuous'), ('momentum', 'long', 'continuous'), ('momentum', 'long', 'discrete')),
    (('volatility', 'both', 'continuous'), ('trend_strength', 'both', 'continuous'), ('divergence', 'long', 'discrete')),
    (('trend_strength', 'both', 'continuous'), ('momentum', 'long', 'continuous'), ('divergence', 'long', 'discrete')),
    (('volatility', 'both', 'continuous'), ('momentum', 'long', 'continuous'), ('divergence', 'long', 'discrete')),
]

PATTERNS_2_EXIT_LONG = [
    (('momentum', 'short', 'continuous'), ('momentum', 'short', 'discrete')),
    (('momentum', 'short', 'continuous'), ('divergence', 'short', 'discrete')),
    (('volatility', 'both', 'continuous'), ('momentum', 'short', 'discrete')),
    (('volatility', 'both', 'continuous'), ('divergence', 'short', 'discrete')),
    (('trend_strength', 'both', 'continuous'), ('momentum', 'short', 'discrete')),
    (('trend_strength', 'both', 'continuous'), ('divergence', 'short', 'discrete')),
]
PATTERNS_3_EXIT_LONG = [
    (('momentum', 'short', 'continuous'), ('momentum', 'short', 'continuous'), ('momentum', 'short', 'discrete')),
    (('momentum', 'short', 'continuous'), ('momentum', 'short', 'continuous'), ('divergence', 'short', 'discrete')),
    (('volatility', 'both', 'continuous'), ('trend_strength', 'both', 'continuous'), ('momentum', 'short', 'discrete')),
    (('volatility', 'both', 'continuous'), ('momentum', 'short', 'continuous'), ('momentum', 'short', 'discrete')),
    (('trend_strength', 'both', 'continuous'), ('momentum', 'short', 'continuous'), ('momentum', 'short', 'discrete')),
    (('volatility', 'both', 'continuous'), ('trend_strength', 'both', 'continuous'), ('divergence', 'short', 'discrete')),
    (('trend_strength', 'both', 'continuous'), ('momentum', 'short', 'continuous'), ('divergence', 'short', 'discrete')),
    (('volatility', 'both', 'continuous'), ('momentum', 'short', 'continuous'), ('divergence', 'short', 'discrete')),
]

PATTERNS_2_ENTRY_SHORT = [
    (('momentum', 'short', 'continuous'), ('momentum', 'short', 'discrete')),
    (('momentum', 'short', 'continuous'), ('divergence', 'short', 'discrete')),
    (('volatility', 'both', 'continuous'), ('momentum', 'short', 'discrete')),
    (('volatility', 'both', 'continuous'), ('divergence', 'short', 'discrete')),
    (('trend_strength', 'both', 'continuous'), ('momentum', 'short', 'discrete')),
    (('trend_strength', 'both', 'continuous'), ('divergence', 'short', 'discrete')),
]
PATTERNS_3_ENTRY_SHORT = [
    (('momentum', 'short', 'continuous'), ('momentum', 'short', 'continuous'), ('momentum', 'short', 'discrete')),
    (('momentum', 'short', 'continuous'), ('momentum', 'short', 'continuous'), ('divergence', 'short', 'discrete')),
    (('volatility', 'both', 'continuous'), ('trend_strength', 'both', 'continuous'), ('momentum', 'short', 'discrete')),
    (('volatility', 'both', 'continuous'), ('momentum', 'short', 'continuous'), ('momentum', 'short', 'discrete')),
    (('trend_strength', 'both', 'continuous'), ('momentum', 'short', 'continuous'), ('momentum', 'short', 'discrete')),
    (('volatility', 'both', 'continuous'), ('trend_strength', 'both', 'continuous'), ('divergence', 'short', 'discrete')),
    (('trend_strength', 'both', 'continuous'), ('momentum', 'short', 'continuous'), ('divergence', 'short', 'discrete')),
    (('volatility', 'both', 'continuous'), ('momentum', 'short', 'continuous'), ('divergence', 'short', 'discrete')),
]

PATTERNS_2_EXIT_SHORT = [
    (('momentum', 'long', 'continuous'), ('momentum', 'long', 'discrete')),
    (('momentum', 'long', 'continuous'), ('divergence', 'long', 'discrete')),
    (('volatility', 'both', 'continuous'), ('momentum', 'long', 'discrete')),
    (('volatility', 'both', 'continuous'), ('divergence', 'long', 'discrete')),
    (('trend_strength', 'both', 'continuous'), ('momentum', 'long', 'discrete')),
    (('trend_strength', 'both', 'continuous'), ('divergence', 'long', 'discrete')),
]
PATTERNS_3_EXIT_SHORT = [
    (('momentum', 'long', 'continuous'), ('momentum', 'long', 'continuous'), ('momentum', 'long', 'discrete')),
    (('momentum', 'long', 'continuous'), ('momentum', 'long', 'continuous'), ('divergence', 'long', 'discrete')),
    (('volatility', 'both', 'continuous'), ('trend_strength', 'both', 'continuous'), ('momentum', 'long', 'discrete')),
    (('volatility', 'both', 'continuous'), ('momentum', 'long', 'continuous'), ('momentum', 'long', 'discrete')),
    (('trend_strength', 'both', 'continuous'), ('momentum', 'long', 'continuous'), ('momentum', 'long', 'discrete')),
    (('volatility', 'both', 'continuous'), ('trend_strength', 'both', 'continuous'), ('divergence', 'long', 'discrete')),
    (('trend_strength', 'both', 'continuous'), ('momentum', 'long', 'continuous'), ('divergence', 'long', 'discrete')),
    (('volatility', 'both', 'continuous'), ('momentum', 'long', 'continuous'), ('divergence', 'long', 'discrete')),
]

# ----------------------------------------------------------------------
# 4. Utility functions for signals and constraints
# ----------------------------------------------------------------------
def get_signals(indicators):
    signals = []
    for ind in indicators:
        ind_name = ind.__class__.__name__
        for method_name in dir(ind):
            method = getattr(ind, method_name)
            if callable(method) and hasattr(method, '_signal_meta'):
                meta = method._signal_meta
                signals.append({
                    'indicator': ind,
                    'indicator_name': ind_name,
                    'method_name': method_name,
                    'category': ind.category,
                    'direction': meta['direction'],
                    'type': meta['type'],
                })
    return signals

def build_signal_pool(indicators):
    pool = defaultdict(list)
    for s in get_signals(indicators):
        key = (s['category'], s['direction'], s['type'])
        pool[key].append((s['indicator'], s['method_name']))
    return pool

def divergence_map_from_indicators(indicators):
    dm = defaultdict(list)
    for ind in indicators:
        base = ind.__class__.__name__
        for method_name in dir(ind):
            if 'divergence' in method_name.lower():
                dm[base].append(method_name)
    return dm

def has_divergence_conflict(selected_methods, divergence_map):
    regular_bases = set()
    for ind, meth in selected_methods:
        base = ind.__class__.__name__
        is_div = any(meth == d for d in divergence_map.get(base, []))
        if not is_div:
            regular_bases.add(base)
    for ind, meth in selected_methods:
        base = ind.__class__.__name__
        is_div = any(meth == d for d in divergence_map.get(base, []))
        if is_div and base in regular_bases:
            return True
    return False

# ----------------------------------------------------------------------
# 5. Random component & strategy generation
# ----------------------------------------------------------------------
def random_component(comp_type, pool, used_indicators=None, divergence_map=None):
    if comp_type == 'entry_long':
        patterns_2, patterns_3 = PATTERNS_2_ENTRY_LONG, PATTERNS_3_ENTRY_LONG
    elif comp_type == 'exit_long':
        patterns_2, patterns_3 = PATTERNS_2_EXIT_LONG, PATTERNS_3_EXIT_LONG
    elif comp_type == 'entry_short':
        patterns_2, patterns_3 = PATTERNS_2_ENTRY_SHORT, PATTERNS_3_ENTRY_SHORT
    elif comp_type == 'exit_short':
        patterns_2, patterns_3 = PATTERNS_2_EXIT_SHORT, PATTERNS_3_EXIT_SHORT
    else:
        raise ValueError("Invalid component type")

    # randomly choose 2 or 3 signals (50% each)
    if random.random() < 0.5:
        pattern = random.choice(patterns_2)
    else:
        pattern = random.choice(patterns_3)

    used_local = set()
    result = []
    for spec in pattern:
        candidates = pool.get(spec, [])
        if not candidates:
            return None
        available = [(ind, name) for (ind, name) in candidates
                     if ind not in used_local and
                     (used_indicators is None or ind not in used_indicators)]
        if not available:
            return None
        ind, name = random.choice(available)
        result.append((ind, name))
        used_local.add(ind)
    # Check divergence conflict within this component
    if has_divergence_conflict(result, divergence_map):
        return None
    return result

def random_strategy(indicators, pool, divergence_map, max_attempts=20):
    for _ in range(max_attempts):
        used_indicators = set()
        el = random_component('entry_long', pool, used_indicators, divergence_map)
        if el is None: continue
        used_indicators.update(ind for ind, _ in el)
        xl = random_component('exit_long', pool, used_indicators, divergence_map)
        if xl is None: continue
        used_indicators.update(ind for ind, _ in xl)
        es = random_component('entry_short', pool, used_indicators, divergence_map)
        if es is None: continue
        used_indicators.update(ind for ind, _ in es)
        xs = random_component('exit_short', pool, used_indicators, divergence_map)
        if xs is None: continue
        # optional global usage check (max 2) is already enforced incrementally
        return {
            'entry_long': [f"{ind.__class__.__name__}.{name}" for ind, name in el],
            'exit_long':  [f"{ind.__class__.__name__}.{name}" for ind, name in xl],
            'entry_short':[f"{ind.__class__.__name__}.{name}" for ind, name in es],
            'exit_short': [f"{ind.__class__.__name__}.{name}" for ind, name in xs],
        }
    raise RuntimeError("Could not generate random strategy after max attempts")

# ----------------------------------------------------------------------
# 6. Fast backtest components (Numba)
# ----------------------------------------------------------------------
@numba.njit(cache=True)
def fast_positions(entry_long, exit_long, entry_short, exit_short):
    n = len(entry_long)
    pos = np.zeros(n, dtype=np.int8)
    current = 0
    for i in range(n):
        if current == 0:
            if entry_long[i]:
                current = 1
            elif entry_short[i]:
                current = -1
        elif current == 1:
            if exit_long[i]:
                current = 0
        elif current == -1:
            if exit_short[i]:
                current = 0
        pos[i] = current
    return pos

def sharpe_and_return_from_positions(positions, returns):
    # positions: array of -1,0,1 (int8), already aligned
    # returns: daily returns (close/close-1) array same length
    strat_returns = np.roll(positions, 1) * returns
    strat_returns[0] = 0
    strat_returns = strat_returns[~np.isnan(strat_returns)]
    if len(strat_returns) == 0:
        return -np.inf, 0
    total_return = np.prod(1 + strat_returns) - 1
    if np.std(strat_returns) == 0:
        return -np.inf, total_return
    sharpe = np.sqrt(252) * np.mean(strat_returns) / np.std(strat_returns)
    return sharpe, total_return

# ----------------------------------------------------------------------
# 7. Pre‑compute raw signals (calls each indicator method once)
# ----------------------------------------------------------------------
def precompute_raw_signals(indicators, data):
    """
    Returns dict: "Indicator.method_name" -> numpy array of int8 (-1,0,1)
    Also returns metadata dict for each method: direction, type, category.
    """
    raw = {}
    meta = {}
    for ind in indicators:
        ind_name = ind.__class__.__name__
        for method_name in dir(ind):
            method = getattr(ind, method_name)
            if callable(method) and hasattr(method, '_signal_meta'):
                # compute the signal array once (must use ind.data, but we pass data)
                # In real code, method() should already use self.data stored inside indicator.
                arr = method()   # assumes method uses its own data
                if isinstance(arr, pd.Series):
                    arr = arr.values
                raw[f"{ind_name}.{method_name}"] = arr.astype(np.int8)
                meta[f"{ind_name}.{method_name}"] = {
                    'direction': method._signal_meta['direction'],
                    'type': method._signal_meta['type'],
                }
    return raw, meta

def binary_signal_from_raw(method_str, role, raw, meta):
    """Return boolean array (0/1) for given role."""
    arr = raw[method_str]
    direction = meta[method_str]['direction']
    if role in ('entry_long', 'exit_short'):
        if direction == 'long':
            return arr == 1
        elif direction == 'both':
            return arr == 1
        else:
            return np.zeros(len(arr), dtype=bool)
    elif role in ('exit_long', 'entry_short'):
        if direction == 'short':
            return arr == -1
        elif direction == 'both':
            return arr == 1
        else:
            return np.zeros(len(arr), dtype=bool)
    else:
        return np.zeros(len(arr), dtype=bool)

def evaluate_strategy_on_chunk(strategy, raw, meta, returns, chunk_slice):
    """Compute Sharpe and total return on a data slice."""
    # Build boolean arrays for the chunk
    el_bool = np.ones(len(returns), dtype=bool)
    for sig in strategy['entry_long']:
        if sig not in raw:
            return -np.inf, 0
        el_bool &= binary_signal_from_raw(sig, 'entry_long', raw, meta)
    xl_bool = np.ones(len(returns), dtype=bool)
    for sig in strategy['exit_long']:
        if sig not in raw:
            return -np.inf, 0
        xl_bool &= binary_signal_from_raw(sig, 'exit_long', raw, meta)
    es_bool = np.ones(len(returns), dtype=bool)
    for sig in strategy['entry_short']:
        if sig not in raw:
            return -np.inf, 0
        es_bool &= binary_signal_from_raw(sig, 'entry_short', raw, meta)
    xs_bool = np.ones(len(returns), dtype=bool)
    for sig in strategy['exit_short']:
        if sig not in raw:
            return -np.inf, 0
        xs_bool &= binary_signal_from_raw(sig, 'exit_short', raw, meta)

    # Slice
    el = el_bool[chunk_slice]
    xl = xl_bool[chunk_slice]
    es = es_bool[chunk_slice]
    xs = xs_bool[chunk_slice]
    rets = returns[chunk_slice]

    positions = fast_positions(el, xl, es, xs)
    return sharpe_and_return_from_positions(positions, rets)

# ----------------------------------------------------------------------
# 8. Walk‑forward testing with early discard
# ----------------------------------------------------------------------
def walk_forward_test(strategy, raw, meta, returns, chunks, min_sharpe_list, min_return_list):
    """
    chunks: list of slice objects or index ranges (e.g., [slice(0,1000), slice(1000,2000), ...])
    min_sharpe_list, min_return_list: same length as chunks
    Returns (pass, final_sharpe, final_return) or (False, None, None)
    """
    for i, chunk in enumerate(chunks):
        sharpe, ret = evaluate_strategy_on_chunk(strategy, raw, meta, returns, chunk)
        if sharpe < min_sharpe_list[i] or ret < min_return_list[i]:
            return False, None, None
    # Passed all chunks
    # Return the performance on the last chunk (or overall)
    last_sharpe, last_ret = evaluate_strategy_on_chunk(strategy, raw, meta, returns, chunks[-1])
    return True, last_sharpe, last_ret

# ----------------------------------------------------------------------
# 9. Main pipeline
# ----------------------------------------------------------------------
def main_pipeline(indicators, data, 
                  n_random_strategies=100000,     # number to generate for pre‑screen
                  top_n_candidates=1000,          # keep best N after pre‑screen
                  pre_screen_bars=2000,           # bars used for pre‑screen (first chunk)
                  pre_screen_min_sharpe=0.5,      # minimum Sharpe to pass pre‑screen
                  pre_screen_min_return=0.0,      # minimum total return
                  walk_forward_chunks=None,       # list of slices (if None, auto‑create equal chunks)
                  min_sharpe_chunks=[0.8, 0.6, 0.6], # per chunk thresholds
                  min_return_chunks=[0.02, 0.01, 0.01],
                  output_file="passing_strategies.jsonl",
                  verbose=True):
    """
    Full pipeline.
    """
    # 1. Pre‑compute raw signals and returns
    raw, meta = precompute_raw_signals(indicators, data)
    returns = data['Close'].pct_change().fillna(0).values

    # 2. Build signal pool and divergence map for random generation
    pool = build_signal_pool(indicators)
    div_map = divergence_map_from_indicators(indicators)

    # 3. Pre‑screen: generate many random strategies, test on first chunk
    first_chunk = slice(0, pre_screen_bars)
    candidates = []
    print(f"Generating {n_random_strategies} random strategies and pre‑screening...")
    pbar = tqdm(range(n_random_strategies))
    for _ in pbar:
        try:
            strat = random_strategy(indicators, pool, div_map)
            sharpe, ret = evaluate_strategy_on_chunk(strat, raw, meta, returns, first_chunk)
            if sharpe >= pre_screen_min_sharpe and ret >= pre_screen_min_return:
                candidates.append((sharpe, strat))
        except RuntimeError:
            continue
        pbar.set_postfix(kept=len(candidates))
    # Sort and keep top N
    candidates.sort(key=lambda x: x[0], reverse=True)
    candidates = candidates[:top_n_candidates]
    print(f"Pre‑screen kept {len(candidates)} strategies")

    # 4. Prepare walk‑forward chunks
    if walk_forward_chunks is None:
        # create equal‑sized chunks covering the rest of data after pre_screen_bars
        remaining = len(data) - pre_screen_bars
        chunk_size = remaining // len(min_sharpe_chunks)
        walk_forward_chunks = []
        start = pre_screen_bars
        for i in range(len(min_sharpe_chunks)):
            end = min(start + chunk_size, len(data))
            walk_forward_chunks.append(slice(start, end))
            start = end
    else:
        # use provided slices
        pass

    # 5. Walk‑forward test each candidate
    passing_strategies = []
    for sharpe, strat in tqdm(candidates, desc="Walk‑forward testing"):
        passed, final_sharpe, final_ret = walk_forward_test(
            strat, raw, meta, returns, walk_forward_chunks, min_sharpe_chunks, min_return_chunks
        )
        if passed:
            record = {
                'strategy': strat,
                'pre_screen_sharpe': sharpe,
                'final_sharpe': final_sharpe,
                'final_return': final_ret
            }
            passing_strategies.append(record)

    # 6. Save results
    with open(output_file, 'w') as f:
        for rec in passing_strategies:
            f.write(json.dumps(rec) + '\n')
    print(f"Saved {len(passing_strategies)} passing strategies to {output_file}")
    return passing_strategies

# ----------------------------------------------------------------------
# 10. Example usage (replace with your real data and indicators)
# ----------------------------------------------------------------------
if __name__ == "__main__":
    # Load your EUR/USD 15‑minute data (example)
    # data = pd.read_csv('eurusd_15m.csv', index_col=0, parse_dates=True)
    # For demo, create random data
    import pandas as pd
    dates = pd.date_range('2006-01-01', '2023-12-31', freq='15T')
    data = pd.DataFrame(index=dates)
    data['Close'] = 1.0 + np.cumsum(np.random.randn(len(dates))*0.0001)
    data['Open'] = data['Close'] * (1 + np.random.randn(len(dates))*0.00005)
    data['High'] = data[['Open','Close']].max(axis=1) + np.abs(np.random.randn(len(dates))*0.0001)
    data['Low']  = data[['Open','Close']].min(axis=1) - np.abs(np.random.randn(len(dates))*0.0001)

    # Instantiate your real indicators (example with mock classes)
    rsi = RSI(data)
    stoch = Stochastic(data)
    adx = ADX(data)
    gk = GarmanKlass(data)
    indicators = [rsi, stoch, adx, gk]  # add all 40 (10 each) as needed

    # Run pipeline
    passing = main_pipeline(
        indicators, data,
        n_random_strategies=10000,       # adjust to your computational budget
        top_n_candidates=500,
        pre_screen_bars=5000,
        pre_screen_min_sharpe=0.5,
        pre_screen_min_return=0.01,
        walk_forward_chunks=None,        # auto chunks
        min_sharpe_chunks=[0.8, 0.7, 0.6],
        min_return_chunks=[0.02, 0.01, 0.005],
        output_file="eurusd_pass.jsonl"
    )
    print(f"Found {len(passing)} strategies that survived walk‑forward")